# 验证 · CoLaR / LT-Tuning / Latent-SFT 三个 code 训练流程(一个 Colab)
**运行时→A100**。三段, **一段一段顺序跑**(不用 Run-all 一次到底更稳):
1. **CoLaR**(transformers 4.45.2)  2. **LT-Tuning**(4.55.4)  3. **Latent-SFT**(4.51.1)

> 三平台 dep 互斥,但**每段 SETUP 会重装自己的 floor**,且训练都跑在 subprocess(`run.py`/`deepspeed`/`bash`)里——子进程读当下装好的版本,所以中途换 transformers 版本**安全**,无需重启运行时。
> 每段小规模(200 行)只验**流程跑通 + loss 收敛**(golden rule)。真训练用 `handoff/train_*.sh`(去 VERIFY, 全量出 ckpt)。

# 验证 · CoLaR-code 训练流程(小规模 + loss 收敛)
**A100 → Run all**。小规模(200 行 · 3ep)验证官方 run.py 训练流程正确、loss 下降。真训练用 `handoff/train_colar_code.sh`。

In [ ]:
# SETUP + 小数据(transformers 4.45.2 官方 CoLaR)
import os, subprocess, torch, json, random
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run('pip -q install "transformers==4.45.2" "lightning==2.5.1.post0" "peft==0.15.2" "omegaconf==2.3.0" "numpy>=2.0,<2.3" sentencepiece accelerate', shell=True)
subprocess.run('pip -q install --force-reinstall --no-deps "huggingface_hub==0.34.4"', shell=True)
for r,u in [("colar","https://github.com/xiaomi-research/colar.git"),("lrm","https://github.com/ruijiezh67/LRM_colab_tasks.git")]:
    if not os.path.exists(f"/content/{r}/.git"): subprocess.run(f"git clone -q {u} /content/{r}", shell=True)
WS="/content/ws"; os.makedirs(f"{WS}/models/llms", exist_ok=True); os.makedirs(f"{WS}/datasets/text_reasoning/coding_mix", exist_ok=True); os.environ["WS"]=WS
LL=f"{WS}/models/llms/Llama-3.2-1B-Instruct"
if not os.path.exists(LL+"/config.json"): subprocess.run(f'huggingface-cli download unsloth/Llama-3.2-1B-Instruct --local-dir {LL} --exclude "original/*"', shell=True, check=True)
GSM="/content/colar_hf/logs/colar/qsa-gsm/colar-final/checkpoints/colar_best.ckpt"
if not os.path.exists(GSM): subprocess.run('huggingface-cli download AlbertTan/CoLaR logs/colar/qsa-gsm/colar-final/checkpoints/colar_best.ckpt --local-dir /content/colar_hf', shell=True, check=True)
os.environ["GSM"]=GSM
tr=json.load(open("/content/lrm/code_real_ladder/colar_train.json")); random.seed(0); random.shuffle(tr); tr=tr[:200]
va=json.load(open("/content/lrm/code_real_ladder/colar_val.json"))[:40]
for nm,dd in [("train",tr),("val",va),("test",va)]: json.dump(dd, open(f"{WS}/datasets/text_reasoning/coding_mix/{nm}.json","w"), ensure_ascii=False, indent=1)
print("SETUP OK | 验证子集 train",len(tr),"val",len(va))

In [ ]:
# 小规模训练 + loss 收敛画图
import os
os.system("pkill -f 'run.py' 2>/dev/null; sleep 2")
WS=os.environ["WS"]; GSM=os.environ["GSM"]
!cd /content/colar && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python -u run.py --model=colar --dataset=qsa --devices=0 --workspace_path={WS} --load_ckpt_path={GSM} --log_suffix=verify --seed=0 dataset_name=coding_mix model_id=Llama-3.2-1B-Instruct batch_size=4 accumulate_grad_batches=2 max_epochs=3 check_val_every_n_epoch=1 2>&1 | tee /content/train.log

import re, glob, matplotlib.pyplot as plt
def show_loss(logpath, title):
    txt = open(logpath, encoding="utf-8", errors="ignore").read()
    ls = [float(x) for x in re.findall(r"(?:'loss'|train_loss|loss)[=:'\s]+([0-9]+\.[0-9]+)", txt)]
    ls = [x for x in ls if x < 1e4]
    if len(ls) >= 2:
        plt.figure(figsize=(6,3)); plt.plot(ls, marker="."); plt.title(title+" · loss"); plt.xlabel("log step"); plt.ylabel("loss"); plt.grid(alpha=.3); plt.show()
        print(f"loss: {ls[0]:.3f} -> {ls[-1]:.3f} ({len(ls)} 点)")
        print("✅ [PASS] 训练流程正确: loss 下降(收敛趋势)" if ls[-1] < ls[0] else "⚠ [WARN] 跑通但 loss 没降 → 查 lr/数据/配方")
    else:
        print("⚠ 没抓到 loss 序列 → 看上面日志确认训练是否真启动")

show_loss("/content/train.log", "CoLaR-code")

# 验证 · LT-Tuning code 训练流程(小规模 + loss 收敛)
**A100 → Run all**。小规模(200 行 · 1ep/阶段)验证官方 run.py 3 阶段课程、loss 下降。真训练用 `handoff/train_lt_code.sh`。

In [ ]:
# SETUP + 小数据 + config(transformers 4.55.4 / deepspeed 0.18.3)
import os, subprocess, pathlib, json, random
subprocess.run('pip -q install "torch==2.7.1" "torchvision==0.22.1" "transformers==4.55.4" "datasets==4.2.0" "deepspeed==0.18.3" "peft==0.18.0" omegaconf accelerate', shell=True)
LT="/content/Latent-Thoughts-Tuning"
if not os.path.exists(LT+"/.git"): subprocess.run(f"git clone -q https://github.com/NeosKnight233/Latent-Thoughts-Tuning.git {LT}", shell=True)
subprocess.run(f"git -C {LT} checkout -q c18aac6", shell=True)
if not os.path.exists("/content/lrm/.git"): subprocess.run("git clone -q https://github.com/ruijiezh67/LRM_colab_tasks.git /content/lrm", shell=True)
# model.py sdpa + alias
mp=pathlib.Path(LT,"model.py"); s=mp.read_text(encoding="utf-8")
s=s.replace('attn_implementation = kwargs.pop("attn_implementation", "flash_attention_2")','attn_implementation = kwargs.pop("attn_implementation", "sdpa")')
if "SoftSeft = LT_Tuning_Model" not in s: s=s.rstrip()+"\n\nSoftSeft = LT_Tuning_Model\n"
mp.write_text(s,encoding="utf-8")
# 小数据
DD="/content/data_lt"; os.makedirs(DD, exist_ok=True)
tr=[json.loads(l) for l in open("/content/lrm/code_real_ladder/lt_train.jsonl",encoding="utf-8") if l.strip()]; random.seed(0); random.shuffle(tr); tr=tr[:200]
va=[json.loads(l) for l in open("/content/lrm/code_real_ladder/lt_val.jsonl",encoding="utf-8") if l.strip()][:40]
open(f"{DD}/lt_code_train.jsonl","w",encoding="utf-8").write("\n".join(json.dumps(r,ensure_ascii=False) for r in tr))
open(f"{DD}/lt_code_val.jsonl","w",encoding="utf-8").write("\n".join(json.dumps(r,ensure_ascii=False) for r in va))
cfg=f"""attn_implementation: sdpa
bf16: true
eval_stage_mode: soft_fusion
eval_strategy: 'no'
fusion_alpha: [0.5, 0.5, 0.6]
fusion_temperature: 1.0
fusion_top_p: 0.9
gradient_accumulation_steps: 4
labels_per_stage: [0, 10, 16]
learning_rate: 5.0e-05
load_model_path: null
logging_steps: 2
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
name: qwen_code
num_train_epochs: 1
output_dir: /content/lt_code_out
per_device_train_batch_size: 4
per_device_eval_batch_size: 4
project: LT_Tuning
remove_unused_columns: true
report_to: none
reset_optimizer: true
resume: 0
save_safetensors: false
save_strategy: 'no'
seed: 42
stage_epochs: [1, 1, 1]
stage_modes: [common, hidden_state, soft_fusion]
stage_names: [stage0-cot, stage1-hidden-state, stage2-soft-fusion]
thinking_hidden_state_layer: -1
thinking_insertion_prob: [0.0, 0.85, 0.95]
thinking_mlp_activation: gelu
thinking_mlp_hidden_dim: 1024
thinking_operator_regex: '[0-9]+|[a-zA-Z_][a-zA-Z0-9_]*'
thinking_secondary_insertion_prob: [0.0, 0.15, 0.2]
thinking_strategy: confidence
thinking_token: <thinking>
thinking_use_mlp: false
train_path: {DD}/lt_code_train.jsonl
use_flash_attention: false
use_unk_for_thinking: false
val_path: {DD}/lt_code_val.jsonl
warmup_ratio: 0.05
weight_decay: 0.01
"""
pathlib.Path(LT,"configs").mkdir(exist_ok=True); pathlib.Path(LT,"configs","qwen_code.yaml").write_text(cfg,encoding="utf-8")
print("SETUP OK | 验证子集 train",len(tr),"| config written")

In [ ]:
# 小规模训练(deepspeed run.py 3 阶段)+ loss 收敛画图
!cd /content/Latent-Thoughts-Tuning && deepspeed --num_gpus 1 run.py configs/qwen_code.yaml 2>&1 | tee /content/train.log

import re, glob, matplotlib.pyplot as plt
def show_loss(logpath, title):
    txt = open(logpath, encoding="utf-8", errors="ignore").read()
    ls = [float(x) for x in re.findall(r"(?:'loss'|train_loss|loss)[=:'\s]+([0-9]+\.[0-9]+)", txt)]
    ls = [x for x in ls if x < 1e4]
    if len(ls) >= 2:
        plt.figure(figsize=(6,3)); plt.plot(ls, marker="."); plt.title(title+" · loss"); plt.xlabel("log step"); plt.ylabel("loss"); plt.grid(alpha=.3); plt.show()
        print(f"loss: {ls[0]:.3f} -> {ls[-1]:.3f} ({len(ls)} 点)")
        print("✅ [PASS] 训练流程正确: loss 下降(收敛趋势)" if ls[-1] < ls[0] else "⚠ [WARN] 跑通但 loss 没降 → 查 lr/数据/配方")
    else:
        print("⚠ 没抓到 loss 序列 → 看上面日志确认训练是否真启动")

show_loss("/content/train.log", "LT-code")

# 验证 · Latent-SFT code 训练流程(小规模 6 步 + loss 收敛)
**A100 → Run all**。小规模(200 行 · S1=2/S2=3ep)验证 6 步管线跑通、stage2 loss 下降。真训练用 `handoff/train_lsft_code.sh`。
> 6 步串行, 含'老代码跑新 Colab'兼容补丁(scatter dtype / force-llama / tokenizer / config)。

In [ ]:
# 验证走 handoff 脚本 VERIFY 模式(6 步太长, 复用已推 mirror 的 .sh; 之后画 loss)
import os, subprocess
if not os.path.exists("/content/lrm/.git"): subprocess.run("git clone -q %s /content/lrm" % "https://github.com/ruijiezh67/LRM_colab_tasks.git", shell=True)
os.environ["WORK"]="/content/crux_retrain_work"; os.environ["VERIFY"]="1"
print(">>> 跑 train_lsft_code.sh VERIFY=1(自包含: floor+6步小规模)。日志 tee /content/train.log ...")
!cd /content && VERIFY=1 WORK=/content/crux_retrain_work bash /content/lrm/handoff/train_lsft_code.sh 2>&1 | tee /content/train.log


In [ ]:
# stage2 loss 收敛画图

import re, glob, matplotlib.pyplot as plt
def show_loss(logpath, title):
    txt = open(logpath, encoding="utf-8", errors="ignore").read()
    ls = [float(x) for x in re.findall(r"(?:'loss'|train_loss|loss)[=:'\s]+([0-9]+\.[0-9]+)", txt)]
    ls = [x for x in ls if x < 1e4]
    if len(ls) >= 2:
        plt.figure(figsize=(6,3)); plt.plot(ls, marker="."); plt.title(title+" · loss"); plt.xlabel("log step"); plt.ylabel("loss"); plt.grid(alpha=.3); plt.show()
        print(f"loss: {ls[0]:.3f} -> {ls[-1]:.3f} ({len(ls)} 点)")
        print("✅ [PASS] 训练流程正确: loss 下降(收敛趋势)" if ls[-1] < ls[0] else "⚠ [WARN] 跑通但 loss 没降 → 查 lr/数据/配方")
    else:
        print("⚠ 没抓到 loss 序列 → 看上面日志确认训练是否真启动")

import glob
logs=glob.glob("/content/crux_retrain_work/lsft_run_distill_stage2*.log")
show_loss(logs[-1] if logs else "/content/train.log", "LSFT-code stage2")